# Kaggle one-cell setup cho Ollama (Mode A + Mode B)

- Cell 1: Mode A (Kaggle goi Remote Ollama Endpoint)
- Cell 2: Mode B (Local Ollama Runtime trong Kaggle)

Ban chay dung 1 cell tuy theo mode ban muon.

In [ ]:
# MODE A: Kaggle -> Remote Ollama Endpoint (khuyen nghi)
# Copy/chay nguyen cell nay trong Kaggle

import os
import pathlib
import subprocess
import yaml

# ====== CONFIG ======
GITHUB_REPO = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'
BRANCH = 'main'
PROJECT_DIR = pathlib.Path('/kaggle/working/llm-convrec')
REMOTE_OLLAMA_BASE_URL = 'http://<YOUR_PUBLIC_OR_TUNNELED_HOST>:11434'
OLLAMA_MODEL = 'llama3.2:3b'

# ====== GET SOURCE (GitHub) ======
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, GITHUB_REPO, str(PROJECT_DIR)], check=True)

# Neu ban dung Kaggle Dataset zip thay vi GitHub, thi bo block clone tren
# va dat PROJECT_DIR tro den thu muc da unzip.

# ====== INSTALL ======
subprocess.run(['pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

# ====== ENV ======
os.environ['OLLAMA_BASE_URL'] = REMOTE_OLLAMA_BASE_URL

# ====== PATCH CONFIG ======
config_path = PROJECT_DIR / 'system_config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['MODEL_PROVIDER'] = 'ollama'
cfg['MODEL'] = OLLAMA_MODEL
cfg['OLLAMA_BASE_URL'] = REMOTE_OLLAMA_BASE_URL

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

# ====== SMOKE TEST (1-turn inference) ======
subprocess.run([
    'python',
    str(PROJECT_DIR / 'smoke_test_inference.py'),
    '--provider', 'ollama',
    '--model', OLLAMA_MODEL
], check=True)

print('Mode A setup + smoke test done.')
print('Neu muon chay full chatbot:')
print('  python /kaggle/working/llm-convrec/clothing_main.py')
print('  python /kaggle/working/llm-convrec/restaurant_main.py')

In [ ]:
# MODE B: Local Ollama Runtime trong Kaggle (de test nhanh, kem on dinh hon Mode A)
# Copy/chay nguyen cell nay trong Kaggle

import os
import pathlib
import subprocess
import time
import yaml

# ====== CONFIG ======
GITHUB_REPO = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'
BRANCH = 'main'
PROJECT_DIR = pathlib.Path('/kaggle/working/llm-convrec')
LOCAL_OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL = 'llama3.2:3b'

# ====== GET SOURCE (GitHub) ======
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, GITHUB_REPO, str(PROJECT_DIR)], check=True)

# ====== INSTALL PROJECT DEPENDENCIES ======
subprocess.run(['pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

# ====== INSTALL + START OLLAMA ======
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run('nohup ollama serve > /tmp/ollama.log 2>&1 &', shell=True, check=True)
time.sleep(5)
subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)

# ====== ENV ======
os.environ['OLLAMA_BASE_URL'] = LOCAL_OLLAMA_BASE_URL

# ====== PATCH CONFIG ======
config_path = PROJECT_DIR / 'system_config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['MODEL_PROVIDER'] = 'ollama'
cfg['MODEL'] = OLLAMA_MODEL
cfg['OLLAMA_BASE_URL'] = LOCAL_OLLAMA_BASE_URL

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

# ====== SMOKE TEST (1-turn inference) ======
subprocess.run([
    'python',
    str(PROJECT_DIR / 'smoke_test_inference.py'),
    '--provider', 'ollama',
    '--model', OLLAMA_MODEL
], check=True)

print('Mode B setup + smoke test done.')
print('Neu muon chay full chatbot:')
print('  python /kaggle/working/llm-convrec/clothing_main.py')
print('  python /kaggle/working/llm-convrec/restaurant_main.py')